# 🏆 Funil Comercial Avançado: Diagnóstico de Conversão com Storytelling

Este notebook apresenta análises de alta profundidade visual e estratégica sobre o pipeline comercial. Focado não apenas no **Quê** (Métricas de Funil), mas no **Por quê** ocorre a perda de negócios e no **Como** escalar resultados baseando-se nos Top Performers.

In [ ]:
import pandas as pd
import json
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# Carregar dados - Base Kaggle (ou Local)
FILE_PATH = '/kaggle/input/datasets/henriqueguedes1977/meetings-sample-v4/meetings_sample.json'
if pd.io.common.file_exists('meetings_sample.json'):
    FILE_PATH = 'meetings_sample.json'

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)

# ==== ENGRENAGEM PRINCIPAL DE PREPARAÇÃO ====
def detectar_etapa(title):
    t = str(title).lower()
    if any(x in t for x in ['assinatura', 'contrato assinado', 'fechamento']):
        return 'Assinatura'
    if any(x in t for x in ['1ª parcela', '1a parcela', 'primeira parcela', 'parcela']):
        return '1ª Parcela'
    if any(x in t for x in ['proposta', 'apresentação proposta', 'proposta comercial']):
        return 'Proposta'
    if any(x in t for x in ['r2', 'reunião 2', 'segunda reunião', '2ª reunião']):
        return 'R2'
    if any(x in t for x in ['r1', 'reunião 1', 'primeira reunião', '1ª reunião', 'cold call']):
        return 'R1'
    return 'Indeterminado'

_ESTADOS = {'AC','AL','AP','AM','BA','CE','DF','ES','GO','MA','MT','MS','MG',
            'PA','PB','PR','PE','PI','RJ','RN','RS','RO','RR','SC','SP','SE','TO'}

def extrair_closer(title):
    partes = [p.strip() for p in str(title).split('-')]
    for p in partes:
        words = p.strip().split()
        if len(words) >= 1 and p.strip().upper() not in _ESTADOS and len(p.strip()) > 3:
            if not p.strip().isupper() or len(words) <= 2:
                return p.strip().title()
    return 'Não Identificado'

df_ef = df.copy()
df_ef['etapa'] = df_ef['title'].apply(detectar_etapa)
df_ef['closer'] = df_ef['title'].apply(extrair_closer)

# Filtrar apenas conteúdo real
mask_efetiva = (df_ef['etapa'] != 'Indeterminado') | (df_ef['SPIN_total'] > 0)
df_ef = df_ef[mask_efetiva].copy()
df_ef.loc[df_ef['etapa'] == 'Indeterminado', 'etapa'] = 'R1'

df_ef['fechou']     = df_ef['etapa'] == 'Assinatura'
df_ef['bonus_parc'] = df_ef['etapa'] == '1ª Parcela'
df_ef['morreu']     = ~df_ef['avancou_funil'] & ~df_ef['fechou']

# Tema corporativo Plotly
CORPORATE_TEMPLATE = 'plotly_white'
MAIN_COLOR = '#0F2027'
ACCENT_COLOR = '#11998e'
ALERT_COLOR = '#e53935'

## 1. Onde Perdemos Nossos Clientes? (Análise de Vazamento do Funil)
Visualizando dinamicamente a quebra volumétrica por etapa. O apontamento identifica o principal abismo onde a receita escapa do caixa.

In [ ]:
n_r1    = int((df_ef['etapa'] == 'R1').sum())
n_r2    = int((df_ef['etapa'] == 'R2').sum())
n_prop  = int((df_ef['etapa'] == 'Proposta').sum())
n_assin = int(df_ef['fechou'].sum())

etv = {'1. Reunião Inicial (R1)': max(n_r1, 1), '2. Qualificação (R2)': n_r2, 
       '3. Apresentação de Proposta': n_prop, '4. Contrato Assinado': n_assin}

fig_funnel = go.Figure(go.Funnel(
    y=list(etv.keys()), 
    x=list(etv.values()),
    textposition='inside', 
    textinfo='value+percent previous',
    marker=dict(color=['#2C3E50', '#34495E', '#18BC9C', '#F39C12']),
    connector={'line': {'color': 'royalblue', 'dash': 'dot', 'width': 3}}
))

# Encontrar onde percentualmente ocorre a maior queda
percent_r2 = n_r2 / max(n_r1, 1)
percent_prop = n_prop / max(n_r2, 1)
maior_queda_idx = 0 if percent_r2 < percent_prop else 1
queda_local = '1 para 2' if maior_queda_idx == 0 else '2 para 3'

# Storytelling Annotation
fig_funnel.add_annotation(
    x=0.5, y=0.5 if maior_queda_idx == 0 else 0.2,
    text=f'🚨 <b>Ponto Crítico de Escape</b><br>Maior taxa de atrito observada na transição da etapa {queda_local}.<br>O lead está esfriando nesta fase.',
    showarrow=True, arrowhead=2, arrowsize=1, arrowwidth=2,
    ax=120, ay=0, font=dict(color=ALERT_COLOR, size=13), bgcolor='rgba(255, 255, 255, 0.9)', bordercolor=ALERT_COLOR
)

fig_funnel.update_layout(
    title=dict(text='<b>Jornada de Conversão: Do Primeiro Contato ao Fechamento</b>', font=dict(size=22)),
    template=CORPORATE_TEMPLATE,
    height=500
)
fig_funnel.show()

## 2. Radiografia de Alta Performance — Quem traz Dinheiro e Por quê?
Uma comparação horizontal que desmascara o mito do volume vs eficiência. Closers que estão acima da Linha de Alta Performance.

In [ ]:
cs = df_ef.groupby('closer').agg(
    volume_total=('fechou','count'), 
    fechados=('fechou','sum'),
    morreram=('morreu','sum'),
    spin_poder=('SPIN_total', 'mean')
).reset_index()

cs['taxa_fechamento'] = (cs['fechados'] / cs['volume_total'] * 100).round(1)
cs = cs[cs['volume_total'] >= 2].sort_values('taxa_fechamento', ascending=True)

fig_closers = go.Figure()
fig_closers.add_trace(go.Bar(
    y=cs['closer'], 
    x=cs['taxa_fechamento'], 
    orientation='h',
    marker=dict(color=cs['taxa_fechamento'], colorscale='Tealgrn', showscale=True),
    text=cs['taxa_fechamento'].apply(lambda x: f'{x}%'),
    textposition='outside',
    hovertext=cs.apply(lambda r: f"Qualidade SPIN Média: {r['spin_poder']:.1f}<br>Total Tentativas: {r['volume_total']}", axis=1)
))

# Storytelling Annotation (Linha de benchmark de 40%)
fig_closers.add_vline(x=40, line_dash="dash", line_color="red", line_width=2)
if len(cs) > 0:
    fig_closers.add_annotation(
        x=41, y=len(cs)-1,
        text="<b>High-Performance Zone (Acima de 40%)</b><br>Equipe aqui domina o gancho de Implicação.",
        showarrow=False, xanchor='left', font=dict(color='red')
    )

fig_closers.update_layout(
    title='<b>Ranking de Eficiência dos Closers (Taxa de Fechamento %)</b>',
    xaxis_title='Taxa de Conversão em %',
    yaxis_title=None,
    template=CORPORATE_TEMPLATE,
    height=max(400, len(cs)*35)
)
fig_closers.show()

## 3. A Matemática da Dor: Correlação entre Método SPIN e Sucesso
Dizemos frequentemente que explorar a dor do cliente fecha contratos, mas o quão verdade é isso? Este gráfico espelha se a profundidade qualitativa da reunião determinou a vitória.

In [ ]:
ef_corr = df_ef.copy()
ef_corr['Resultado Final'] = ef_corr['fechou'].apply(lambda x: '✅ Ganho (Win)' if x else '❌ Perdido (Loss)')

spin_corr = ef_corr.groupby('Resultado Final')[['SPIN_total', 'P_score', 'I_score']].mean().reset_index()

fig_corr = go.Figure()

categories = ['Problem (P_score)', 'Implication (I_score)', 'Total Execution (SPIN_total)']
cores = {'✅ Ganho (Win)': '#11998e', '❌ Perdido (Loss)': '#e53935'}

for res in spin_corr['Resultado Final'].unique():
    dados = spin_corr[spin_corr['Resultado Final'] == res]
    y_vals = [dados['P_score'].values[0], dados['I_score'].values[0], dados['SPIN_total'].values[0]/3]  # Normalizado o Total para caber na escala 0-3 visualmente
    
    fig_corr.add_trace(go.Bar(
        name=res,
        x=categories,
        y=y_vals,
        marker_color=cores[res],
        text=[f"{v:.1f}" for v in y_vals], textposition='auto'
    ))

if len(spin_corr) > 0:
    fig_corr.add_annotation(
        x=1, y=max(spin_corr['I_score'])+0.3,
        text="<b>A Chave do Sucesso</b>:<br>Negócios ganhos geralmente têm score 'I' muito maior.<br>Sem Implicação, o cliente adia a decisão.",
        showarrow=True, arrowhead=1, ax=0, ay=-50
    )

fig_corr.update_layout(
    title="<b>Raio-X da Conversa: Por que negócios fecham? (Médias de Score)</b>",
    barmode='group',
    template=CORPORATE_TEMPLATE
)
fig_corr.show()

## 4. O Resumo Executivo Definitivo do Pipeline

<div style="font-family: Arial, sans-serif; background-color: #f8f9fa; padding: 30px; border-radius: 12px; border-left: 8px solid #0F2027; box-shadow: 0px 4px 10px rgba(0,0,0,0.05);">
<h1 style="color: #0F2027; margin-top: 0;">Diagnóstico Executivo de Alta Conversão B2B</h1>
<p style="color: #555; font-size: 14px; text-transform: uppercase; letter-spacing: 1px;">Documento Estratégico · Departamento Comercial Villela</p>

<hr style="border: 0; border-top: 1px solid #ddd; margin: 20px 0;">

<div style="display: flex; gap: 20px; flex-wrap: wrap;">
<div style="flex: 1; min-width: 250px; background: white; padding: 20px; border-radius: 8px; border-top: 4px solid #F39C12;">
<h3 style="color: #333; margin-top:0;">🛑 1. O Ponto de Sangria do Funil</h3>
<p>Observamos uma disparidade massiva entre as conexões iniciais (R1) e as conclusões das propostas (Assinatura). A maior evasão do nosso funil acontece na incapacidade de <strong>ancorar o prejuízo do cliente no tempo certo</strong>. Sem formalizar o compromisso micro-passo a micro-passo, os leads caem no limbo do "vou pensar".</p>
</div>

<div style="flex: 1; min-width: 250px; background: white; padding: 20px; border-radius: 8px; border-top: 4px solid #18BC9C;">
<h3 style="color: #333; margin-top:0;">🏆 2. O Segredo dos Top Performers</h3>
<p>Existe um limite nítido separando os <i>Closers Tradicionais</i> dos <i>Closers de Elite</i>. Matematicamente, as reuniões concluídas com sucesso pelos High Performers <strong>possuem um volume quase perfeito de extração de dores e de implicações em tempo hábil</strong>. Eles forçam o empresário a dizer em voz alta as consequências jurídicas e financeiras de sua inércia fiscal.</p>
</div>
</div>

<br>
<div style="background: #2C3E50; padding: 25px; border-radius: 8px; color: white;">
<h2 style="color: #f1c40f; margin-top: 0;">⚡ As 5 Diretrizes de Correção de Rota Imediatas</h2>
<ul style="line-height: 1.8; font-size: 16px;">
<li><strong>Obrigatoriedade da Tríade de Implicação:</strong> Proibido saltar para apresentação da solução sem fazer ao menos 3 perguntas focadas em "Riscos e Multas" caso o cliente não resolva sua situação tributária hoje.</li>
<li><strong>Revisão de Qualificação (Gate de R1):</strong> Diversas calls não deveriam acontecer. O SDR deve bloquear R1s de clientes que não expressam um gatilho nítido de dor sistêmica prévia no BANT.</li>
<li><strong>Follow-up Condicionado (SLA 24h):</strong> Assinatura de uma Ata Resumo gerada via IA e enviada ao cliente com os compromissos nos primeiros 20 minutos pós-reunião.</li>
<li><strong>Clínica de Roleplay Semanal:</strong> Duas horas semanais onde o Bottom de Closers treina especificamente os gatilhos difíceis focados na <i>Necessidade</i> (pay-off verbal do risco).</li>
<li><strong>Política de "No-Show Limpo":</strong> O Salesforce/CRM deve abortar e devolver leads sem evolução clara pro espectro de marketing após os 10 dias parados pós-proposta, poupando o Pipeline de ilusões.</li>
</ul>
</div>
</div>
